In [94]:
# Creating bounding box for the object in the first frame
# for ethernet port: (353, 359, 365, 371, 390), (426, 449, 461*, 468*, 471, 496, 515) 
# for power port: (333, 353, 359, 365), (426, 449, 461, 468, 471, 496), (400, 496, 531)
import cv2
import json
import os

# ----------- CONFIG -----------
image_dir = "Data"
image_prefix = "frame_000"
image_ext = ".png"

# Your selected images
# image_indices = [319, 333, 353, 359, 365, 371, 390,426, 449, 461, 468, 471, 496, 515]   # ethernet ports
image_indices = [319, 333, 353, 359, 365, 400,426, 449, 461, 468, 471, 496, 531]   # power ports


# output_file = "bboxes_ethernet_socket.json"
output_file = "bboxes_power_socket.json"
# output_file = "bboxes_vga_socket_tight.json"

# --------------------------------


def get_image_path(i):
    path = image_dir + "/" + image_prefix + str(i) + image_ext
    return path

bboxes = {}

for i in image_indices:
    path = get_image_path(i)
    print(f"\nProcessing image {i}: {path}")

    img = cv2.imread(path)

    if img is None:
        print(f"Could not load image {path}, skipping...")
        continue

    print("Draw bounding box and press ENTER")
    print("Press 'c' to skip")

    bbox = cv2.selectROI(f"Image {i}", img, fromCenter=False, showCrosshair=True)
    cv2.destroyAllWindows()

    if bbox == (0, 0, 0, 0):
        print("Skipped")
        bboxes[str(i)] = None
    else:
        print(f"Saved bbox: {bbox}")
        bboxes[str(i)] = list(map(int, bbox))

# Save to JSON
with open(output_file, "w") as f:
    json.dump(bboxes, f, indent=4)

print(f"\nBounding boxes saved to {output_file}")


Processing image 319: Data/frame_000319.png
Draw bounding box and press ENTER
Press 'c' to skip
Select a ROI and then press SPACE or ENTER button!
Cancel the selection process by pressing c button!
Saved bbox: (1431, 446, 103, 257)

Processing image 333: Data/frame_000333.png
Draw bounding box and press ENTER
Press 'c' to skip
Select a ROI and then press SPACE or ENTER button!
Cancel the selection process by pressing c button!
Saved bbox: (1527, 485, 200, 323)

Processing image 353: Data/frame_000353.png
Draw bounding box and press ENTER
Press 'c' to skip
Select a ROI and then press SPACE or ENTER button!
Cancel the selection process by pressing c button!
Saved bbox: (1102, 593, 350, 518)

Processing image 359: Data/frame_000359.png
Draw bounding box and press ENTER
Press 'c' to skip
Select a ROI and then press SPACE or ENTER button!
Cancel the selection process by pressing c button!
Saved bbox: (1791, 554, 423, 560)

Processing image 365: Data/frame_000365.png
Draw bounding box and p

QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 

Saved bbox: (1853, 561, 229, 383)

Processing image 468: Data/frame_000468.png
Draw bounding box and press ENTER
Press 'c' to skip
Select a ROI and then press SPACE or ENTER button!
Cancel the selection process by pressing c button!
Saved bbox: (1559, 653, 293, 465)

Processing image 471: Data/frame_000471.png
Draw bounding box and press ENTER
Press 'c' to skip
Select a ROI and then press SPACE or ENTER button!
Cancel the selection process by pressing c button!
Saved bbox: (1151, 642, 372, 471)

Processing image 496: Data/frame_000496.png
Draw bounding box and press ENTER
Press 'c' to skip
Select a ROI and then press SPACE or ENTER button!
Cancel the selection process by pressing c button!
Saved bbox: (1721, 695, 180, 470)

Processing image 531: Data/frame_000531.png
Draw bounding box and press ENTER
Press 'c' to skip
Select a ROI and then press SPACE or ENTER button!
Cancel the selection process by pressing c button!
Saved bbox: (879, 91, 751, 1173)

Bounding boxes saved to bboxes_pow

In [77]:
# loading the intrinsics

import numpy as np
import json

def load_intrinsics(file_path):
    with open(file_path, "r") as f:
        data = json.load(f)

    K = np.array(data["camera_matrix"], dtype=np.float64)

    # Optional extras (useful later)
    width = data.get("image_width", None)
    height = data.get("image_height", None)
    dist = np.array(data.get("distortion_coefficients", []), dtype=np.float64)

    return K, width, height, dist

K, width, height, dist = load_intrinsics("Data/intrinsic.json")

print("K:\n", K)


K:
 [[1.47700975e+03 0.00000000e+00 1.29825015e+03]
 [0.00000000e+00 1.48044245e+03 6.86820162e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]


In [78]:
# class for managing tracks across frames

class TrackManager:
    def __init__(self, kp_ref, ref_frame_id):
        self.tracks = {}

        # initialize tracks from keyframe
        for i, kp in enumerate(kp_ref):
            self.tracks[i] = [(ref_frame_id, kp.pt)]

    def add_matches(self, matches, kp_curr, curr_frame_id):
        for m in matches:
            ref_id = m.queryIdx
            pt = kp_curr[m.trainIdx].pt

            self.tracks[ref_id].append((curr_frame_id, pt))

    def get_valid_tracks(self, min_length=3):
        return {
            k: v for k, v in self.tracks.items()
            if len(v) >= min_length
        }

In [79]:
# ---------- SIFT with ROI mask ----------

def kaze_in_bbox(img, bbox):
    x, y, w, h = bbox

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    mask = np.zeros(gray.shape, dtype=np.uint8)
    mask[y:y+h, x:x+w] = 255

    kaze = cv2.KAZE_create()
    
    # adding pipeline for refining bbox to OBB and creating mask--------------------------
    # _, mask = refine_bbox_to_obb(img, bbox)
    kp, des = kaze.detectAndCompute(gray, mask)

    return kp, des

def sift_in_bbox(img, bbox):
    x, y, w, h = bbox

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    mask = np.zeros(gray.shape, dtype=np.uint8)
    mask[y:y+h, x:x+w] = 255

    sift = cv2.SIFT_create ()
    kp, des = sift.detectAndCompute(gray, mask)
    # kp, des = sift.detectAndCompute(gray, mask)
    
    # Convert to RootSIFT
    def rootsift(descriptors):
    # L1 normalize
        descriptors /= (descriptors.sum(axis=1, keepdims=True) + 1e-7)
    
        # Square root
        descriptors = np.sqrt(descriptors)
    
        return descriptors

    # des_root = rootsift(des)

    # return kp, des_root 

    return kp, des

# ---------- RANSAC filtering ----------
def ransac_filter(kp1, kp2, matches, K):
    if len(matches) < 8:
        print("Not enough matches for RANSAC")
        return [], None

    pts1 = np.float32([kp1[m.queryIdx].pt for m in matches])
    pts2 = np.float32([kp2[m.trainIdx].pt for m in matches])

    E, mask = cv2.findEssentialMat(
        pts1, pts2, K,
        method=cv2.RANSAC,
        prob=0.999,
        threshold=2.0
    )

    if mask is None:
        print("RANSAC failed")
        return [], None

    inliers = mask.ravel().astype(bool)

    filtered_matches = [m for i, m in enumerate(matches) if inliers[i]]

    return filtered_matches, inliers

# function to match and visualize with RANSAC filtering
def match_and_visualize(img1, kp1, des1, img2, kp2, des2, K):

    if des1 is None or des2 is None:
        print("No descriptors found, skipping...")
        return []

    bf = cv2.BFMatcher()

    matches = bf.knnMatch(des1, des2, k=2)

    # Ratio test
    good_matches = []
    for m, n in matches:
        if m.distance < 0.75 * n.distance:
            good_matches.append(m)

    print(f"Good matches (ratio test): {len(good_matches)}")

    # RANSAC
    inlier_matches, _ = ransac_filter(kp1, kp2, good_matches, K)
    print(f"Inliers after RANSAC: {len(inlier_matches)}")

    if len(inlier_matches) == 0:
        print("No inliers found")
        return []

    # Visualization
    img_match = cv2.drawMatches(
        img1, kp1,
        img2, kp2,
        inlier_matches,
        None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )

    cv2.imshow("RANSAC Matches", img_match)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

    return inlier_matches

In [80]:
import cv2
import numpy as np
import json
import os

# -------- CONFIG --------
image_dir = "Data"
image_prefix = "frame_000"
image_ext = ".png"
file_name_prefix = "bboxes_"
file_name_suffix = ".json"

# device_type = "ethernet_socket"  
device_type = "power_socket"  
# device_type = "vga_socket" 

file_name = file_name_prefix + device_type + file_name_suffix
bbox_file = os.path.join(image_dir, file_name)

image_indices = [319, 333, 353, 359, 365, 371, 390,426, 449, 461, 468, 471, 496, 515]   # ethernet ports
# image_indices = [319, 333, 353, 359, 365, 400,426, 449, 461, 468, 471, 496, 531]   # power ports

# image_indices = [426, 449, 461, 468, 471] #ethernet, power
image_indices = [359, 365, 371] # vga

# -------------get image indices-----------
def get_image_indices(device_type=device_type, frame_set_idx=0):
    if device_type == "ethernet_socket":
        if frame_set_idx == 0:
            image_indices = [426, 449, 461, 468, 471]
            key_frame_idx = 426
        elif frame_set_idx == 1:
            image_indices = [359, 365, 371, 390]
            key_frame_idx = 359
        else:   
            image_indices = [319, 333, 353, 359, 365, 371, 390,426, 449, 461, 468, 471, 496, 515]      
            key_frame_idx = 319
    elif device_type == "power_socket":
        if frame_set_idx == 0:
            image_indices = [426, 449, 461, 468, 471]
            key_frame_idx = 426
        elif frame_set_idx == 1:
            image_indices = [359, 365, 371, 400]
            key_frame_idx = 359
        else:
            image_indices = [319, 333, 353, 359, 365, 400,426, 449, 461, 468, 471, 496, 531]
            key_frame_idx = 319
    elif device_type == "vga_socket":
        if frame_set_idx == 0:
            # image_indices = [359, 365, 371, 390]
            # image_indices = [359, 371]
            image_indices = [359, 365, 371]
            key_frame_idx = 359
        elif frame_set_idx == 1:
            image_indices = [426, 449, 461, 468, 471, 515]
            key_frame_idx = 426
        else:
            image_indices = [333, 353, 359, 365, 400, 426, 449, 461, 468, 471, 496, 531]
            key_frame_idx = 333
    else:
        raise ValueError("Unknown device type")

    # For now we just return all indices for the selected device
    return image_indices, key_frame_idx

def get_image_path(i):
    path = image_dir + "/" + image_prefix + str(i) + image_ext
    return path


# ---------- Load bounding boxes ----------
with open(bbox_file, "r") as f:
    bboxes = json.load(f)

# ---------- Load keyframe once ----------
image_indices, key_frame_idx = get_image_indices(device_type=device_type, frame_set_idx=1)
# key_frame_idx = image_indices[0]

img_ref = cv2.imread(get_image_path(key_frame_idx))
bbox_ref = bboxes.get(str(key_frame_idx))

#---------------- sift in bbox for keyframe
kp_ref, des_ref = sift_in_bbox(img_ref, bbox_ref)

#------------------Kaze in bbox for keyframe
# kp_ref, des_ref = kaze_in_bbox(img_ref, bbox_ref)

print(f"Keyframe keypoints: {len(kp_ref)}")


# ---------- Initialize Track Manager ----------
track_manager = TrackManager(kp_ref, key_frame_idx)


# ---------- Loop over other frames ----------
for idx2 in image_indices[1:]:

    print(f"\nMatching {key_frame_idx} ↔ {idx2}")

    img2 = cv2.imread(get_image_path(idx2))
    bbox2 = bboxes.get(str(idx2))

    if img2 is None or bbox2 is None:
        print("Missing data")
        continue

    kp2, des2 = sift_in_bbox(img2, bbox2)
    # kp2, des2 = kaze_in_bbox(img2, bbox2)

    print(f"Keypoints: {len(kp_ref)} vs {len(kp2)}")

    # -------- Match + RANSAC --------
    inlier_matches = match_and_visualize(
        img_ref, kp_ref, des_ref,
        img2, kp2, des2,
        K
    )

    # -------- Update tracks --------
    track_manager.add_matches(
        inlier_matches,
        kp2,
        idx2
    )

Keyframe keypoints: 123

Matching 359 ↔ 365
Keypoints: 123 vs 138
Good matches (ratio test): 85
Inliers after RANSAC: 82

Matching 359 ↔ 371
Missing data

Matching 359 ↔ 400
Keypoints: 123 vs 28
Good matches (ratio test): 15
Inliers after RANSAC: 12


In [81]:
def visualize_tracks(image_indices, tracks, get_image_path, bboxes=None, max_tracks=30):
    """
    Visualize tracks across frames.

    - Same color = same track
    - Lines show motion across frames
    - max_tracks limits clutter
    """

    # limit number of tracks for clarity
    track_ids = list(tracks.keys())[:max_tracks]

    # assign color per track
    colors = {
        tid: tuple(np.random.randint(0, 255, 3).tolist())
        for tid in track_ids
    }

    for frame_id in image_indices:
        img = cv2.imread(get_image_path(frame_id)).copy()

        # draw bbox if available
        if bboxes is not None:
            bbox = bboxes.get(str(frame_id))
            if bbox is not None:
                x, y, w, h = bbox
                cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 2)

        for tid in track_ids:
            if tid not in tracks:
                continue

            observations = tracks[tid]

            pts = []
            for (f_id, pt) in observations:
                if f_id <= frame_id:
                    pts.append((int(pt[0]), int(pt[1])))

            # draw trajectory
            for i in range(1, len(pts)):
                cv2.line(img, pts[i-1], pts[i], colors[tid], 2)

            # draw current point
            for (f_id, pt) in observations:
                if f_id == frame_id:
                    x, y = int(pt[0]), int(pt[1])
                    cv2.circle(img, (x, y), 5, colors[tid], -1)

        cv2.imshow("Track Visualization", img)

        key = cv2.waitKey(0)
        if key == 27:  # ESC to exit early
            break

    cv2.destroyAllWindows()

In [82]:
file_name_suffix = "_tight.json"
file_name = file_name_prefix + device_type + file_name_suffix
tight_bbox_file = os.path.join(image_dir, file_name)
# print(file_name)
# print(tight_bbox_file)
with open(tight_bbox_file, "r") as f:
    tight_bboxes = json.load(f)
    
def filter_tracks_by_tight_roi(tracks, tight_bboxes, key_frame_idx):
    """
    Keeps only tracks whose keyframe point lies inside tight ROI
    """

    if str(key_frame_idx) not in tight_bboxes:
        raise ValueError("Keyframe ROI not found in tight bbox file")

    x, y, w, h = tight_bboxes[str(key_frame_idx)]

    filtered_tracks = {}

    for track_id, observations in tracks.items():

        # find keyframe observation
        for (frame_id, pt) in observations:
            if frame_id == key_frame_idx:
                px, py = pt

                # check if inside ROI
                if (x <= px <= x + w) and (y <= py <= y + h):
                    filtered_tracks[track_id] = observations

                break  # important: stop after keyframe

    return filtered_tracks

In [83]:
valid_tracks = track_manager.get_valid_tracks(min_length=2)

print("\n--- Tracking Summary ---")
print(f"Total tracks: {len(track_manager.tracks)}")
print(f"Valid tracks (>=3 frames): {len(valid_tracks)}")

for k, v in list(valid_tracks.items())[:5]:  # Print first 5 valid tracks
    print(f"\nTrack {k}:")
    for obs in v:
        print(obs)
print("\nVisualizing tracks...")

visualize_tracks(
    image_indices,
    valid_tracks,
    get_image_path,
    bboxes=bboxes,
    max_tracks=30   # reduce clutter
)

filtered_tracks = filter_tracks_by_tight_roi(
    valid_tracks,
    tight_bboxes,
    key_frame_idx
)

visualize_tracks(
    image_indices,
    filtered_tracks,
    get_image_path,
    bboxes=bboxes,
    max_tracks=30   # reduce clutter
)

print(f"Tracks before filtering: {len(valid_tracks)}")
print(f"Tracks after filtering: {len(filtered_tracks)}")


--- Tracking Summary ---
Total tracks: 123
Valid tracks (>=3 frames): 86

Track 1:
(359, (1820.6324462890625, 582.274658203125))
(365, (1483.8719482421875, 653.9083251953125))

Track 2:
(359, (1821.648681640625, 1025.0225830078125))
(400, (1341.935546875, 887.1422119140625))

Track 3:
(359, (1824.1868896484375, 1020.1405029296875))
(365, (1513.807861328125, 1192.841064453125))
(400, (1354.2152099609375, 883.4557495117188))

Track 6:
(359, (1832.5897216796875, 672.2725830078125))
(365, (1503.78466796875, 765.689453125))

Track 7:
(359, (1832.5897216796875, 672.2725830078125))
(365, (1503.78466796875, 765.689453125))

Visualizing tracks...
Tracks before filtering: 86
Tracks after filtering: 6


In [84]:
import numpy as np
import json

"""

    Naming convention:
    ------------------
    T_ab : transforms a point from frame 'b' to frame 'a'

    So:
    - T_wc : camera → world  (given)
    - T_cw : world → camera  (needed for projection)

    A 3D point transforms as:
        X_w = T_wc @ X_c
        X_c = T_cw @ X_w

    This function computes:
        T_cw = inverse(T_wc)
"""

def load_camera_poses(json_path):
    with open(json_path, 'r') as f:
        poses_data = json.load(f)

    camera_poses = {}

    for frame_id, mat in poses_data.items():
        T = np.array(mat)  # 4x4

        # Extract R and t from T_cw
        R_cw = T[:3, :3]
        t_cw = T[:3, 3].reshape(3,1)

        # Convert to T_wc
        R_wc = R_cw.T
        t_wc = -R_wc @ t_cw
        
        # Extract R and t from T_cw
        # R_wc = T[:3, :3]
        # t_wc = T[:3, 3].reshape(3,1)

        camera_poses[int(frame_id)] = (R_wc, t_wc)

    return camera_poses

In [95]:
def invert_poses(poses_wc):
    """
    Convert camera poses from T_wc to T_cw.

    Naming convention:
    ------------------
    T_ab : transforms a point from frame 'b' to frame 'a'

    So:
    - T_wc : camera → world  (given)
    - T_cw : world → camera  (needed for projection)

    A 3D point transforms as:
        X_w = T_wc @ X_c
        X_c = T_cw @ X_w

    This function computes:
        T_cw = inverse(T_wc)
    """
    poses_cw = []

    for T_wc in poses_wc:
        R_wc = T_wc[:3, :3]
        t_wc = T_wc[:3, 3]

        R_cw = R_wc.T
        t_cw = -R_wc.T @ t_wc

        T_cw = np.eye(4)
        T_cw[:3, :3] = R_cw
        T_cw[:3, 3] = t_cw

        poses_cw.append(T_cw)

    return poses_cw

In [85]:
camera_poses = load_camera_poses("Data/poses.json")
# poses_cw = invert_poses(camera_poses)

# print(camera_poses[0] @ poses_cw[0])

# R, t = camera_poses[426]
# print("\nCamera pose for frame 426:")
# print("R:\n", R)
# print("t:\n", t)
# print("R shape:", R.shape)  # should be (3,3)
# print("t shape:", t.shape)  # should be (3,1)

In [86]:
def triangulate_tracks(tracks, camera_poses, K):
    points_3d = []

    for track_id, track in tracks.items():
        if len(track) < 2:
            continue

        # use first two observations
        (f1, pt1), (f2, pt2) = track[:2]

        R1, t1 = camera_poses[f1]
        R2, t2 = camera_poses[f2]

        P1 = K @ np.hstack((R1, t1))
        P2 = K @ np.hstack((R2, t2))

        pt1 = np.array(pt1).reshape(2,1)
        pt2 = np.array(pt2).reshape(2,1)

        X = cv2.triangulatePoints(P1, P2, pt1, pt2)
        X = (X[:3] / X[3]).ravel()

        if np.isfinite(X).all():
            points_3d.append(X)

    return np.array(points_3d)

In [87]:
print(K)
points_3d = triangulate_tracks(filtered_tracks, camera_poses, K)

print("3D points shape:", points_3d.shape)
print(points_3d)
print("Std dev:", np.std(points_3d, axis=0))
socket_pos = np.mean(points_3d, axis=0)
print("Socket position:", socket_pos)
min_vals = points_3d.min(axis=0)
max_vals = points_3d.max(axis=0)
extent = (max_vals - min_vals)
print("Min:", min_vals)
print("Max:", max_vals)
print("Extent:", extent)
local_pts = (points_3d - socket_pos)

[[1.47700975e+03 0.00000000e+00 1.29825015e+03]
 [0.00000000e+00 1.48044245e+03 6.86820162e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]
3D points shape: (6, 3)
[[0.28290317 0.22349903 0.52717512]
 [0.28291069 0.2286072  0.52664988]
 [0.28803147 0.22574646 0.52348605]
 [0.29145636 0.22655272 0.52332273]
 [0.29539861 0.22933453 0.52046983]
 [0.29660997 0.22936433 0.52671471]]
Std dev: [0.00545054 0.00213885 0.00242308]
Socket position: [0.28955171 0.22718405 0.52463639]
Min: [0.28290317 0.22349903 0.52046983]
Max: [0.29660997 0.22936433 0.52717512]
Extent: [0.0137068  0.0058653  0.00670529]


In [88]:
print("\nVisualizing tracks...")
# visualize_tracks(
#     image_indices,
#     valid_tracks,
#     get_image_path,
#     bboxes=bboxes,
#     max_tracks=70   # reduce clutter
# )
print(f"Tracks before filtering: {len(valid_tracks)}")
print(f"Tracks after filtering: {len(filtered_tracks)}")

points_3d = triangulate_tracks(valid_tracks, camera_poses, K)
# points_3d = triangulate_tracks(filtered_tracks, camera_poses, K)

print("3D points shape:", points_3d.shape)
print(points_3d)
print("Std dev:", np.std(points_3d, axis=0))




Visualizing tracks...
Tracks before filtering: 86
Tracks after filtering: 6
3D points shape: (86, 3)
[[0.26552767 0.22788862 0.71944868]
 [0.28290317 0.22349903 0.52717512]
 [0.28291069 0.2286072  0.52664988]
 [0.27305618 0.22983472 0.68194972]
 [0.27305618 0.22983472 0.68194972]
 [0.28803147 0.22574646 0.52348605]
 [0.27074857 0.22964925 0.72059858]
 [0.29145636 0.22655272 0.52332273]
 [0.27738481 0.2287165  0.6842147 ]
 [0.27742755 0.22856277 0.68672346]
 [0.29065296 0.22347057 0.56424366]
 [0.27690228 0.22985371 0.72961723]
 [0.29539861 0.22933453 0.52046983]
 [0.2817533  0.22917853 0.68554244]
 [0.2818574  0.22907824 0.68833035]
 [0.29660997 0.22936433 0.52671471]
 [0.28186849 0.22891213 0.70179468]
 [0.28186849 0.22891213 0.70179468]
 [0.28702951 0.18880646 0.72892477]
 [0.28474538 0.22886664 0.68626886]
 [0.27430606 0.28346571 0.71195753]
 [0.28273525 0.22709716 0.72565925]
 [0.2843712  0.22829045 0.72729725]
 [0.28833287 0.22813235 0.71162104]
 [0.29074131 0.22921309 0.70202249

In [92]:
# Code for finding the rotation of the socket using PCA on the 3D points

import numpy as np
np.random.seed(42)

def fit_plane(p1, p2, p3):
    v1 = p2 - p1
    v2 = p3 - p1
    normal = np.cross(v1, v2)
    normal /= np.linalg.norm(normal)
    d = -np.dot(normal, p1)
    return normal, d

def point_plane_distance(points, normal, d):
    return np.abs(points @ normal + d)

points_3d = np.unique(points_3d, axis=0)
best_inliers = []
best_model = None

for _ in range(500):
    ids = np.random.choice(len(points_3d), 3, replace=False)
    p1, p2, p3 = points_3d[ids]

    normal, d = fit_plane(p1, p2, p3)

    dist = point_plane_distance(points_3d, normal, d)
    
    threshold = 0.002   # or tuned value
    inliers = points_3d[dist < threshold]

    if len(inliers) > len(best_inliers):
        best_inliers = inliers
        best_model = (normal, d)
        
points_filtered = np.array(best_inliers)

centroid = points_filtered.mean(axis=0)
X = points_filtered - centroid

_, _, Vt = np.linalg.svd(X)

x_axis = Vt[0]
y_axis = Vt[1]
z_axis = Vt[2]

z_axis = 1 * Vt[2] / np.linalg.norm(Vt[2])
if z_axis[1] < 0:
    z_axis = -z_axis
    
x_axis = Vt[0] / np.linalg.norm(Vt[0])

y_axis = np.cross(x_axis, z_axis)
y_axis /= np.linalg.norm(y_axis)

# recompute x to fix drift
x_axis = np.cross(y_axis, z_axis)
x_axis /= np.linalg.norm(x_axis)

if x_axis[2] < 0:
    x_axis = -x_axis
    
Rotation_matrix = np.column_stack((x_axis, y_axis, z_axis))
print("x_axis:", x_axis)
print("y_axis:", y_axis)
print("z_axis:", z_axis)
print("Rotation R:\n", Rotation_matrix)
print("Rotation R:\n", Rotation_matrix[2][0])

# --- Transform to object frame ---
local_pts_here = local_pts
local_pts_here = local_pts_here @ Rotation_matrix
# # --- Robust extent ---
min_vals = local_pts_here.min(axis=0)
max_vals = local_pts_here.max(axis=0)
extent = max_vals - min_vals

print("Extent:", extent)

x_axis: [-0.03877377  0.00967822  0.99920114]
y_axis: [-0.99826172 -0.0447927  -0.03830346]
z_axis: [-0.04438621  0.99894942 -0.01139818]
Rotation R:
 [[-0.03877377 -0.99826172 -0.04438621]
 [ 0.00967822 -0.0447927   0.99894942]
 [ 0.99920114 -0.03830346 -0.01139818]]
Rotation R:
 0.9992011441292874
Extent: [0.00712795 0.01392807 0.00535117]


In [14]:
# Code for writing the OBB to JSON

import json
import numpy as np

def update_entity_obb(json_path, entity_name, center, extent, rotation):
    # Convert to Python lists (important for JSON)
    center = np.asarray(center).tolist()
    extent = np.asarray(extent).tolist()
    rotation = np.asarray(rotation).tolist()

    # Load JSON
    with open(json_path, "r") as f:
        data = json.load(f)

    # Find and update entity
    found = False
    for obj in data:
        if obj["entity"] == entity_name:
            obj["obb"]["center"] = center
            obj["obb"]["extent"] = extent
            obj["obb"]["rotation"] = rotation
            found = True
            break

    if not found:
        raise ValueError(f"Entity '{entity_name}' not found in JSON")

    # Save back
    with open(json_path, "w") as f:
        json.dump(data, f, indent=4)

    print(f"Updated {entity_name} successfully!")

file_dir = "Data"
file_name = "sample_answers.json"
file_path = os.path.join(file_dir, file_name)
# take care of rotation while dumping
# update_entity_obb(
#     file_path,
#     device_type,
#     socket_pos,
#     extent,
#     R
# )

In [93]:
import json
import numpy as np
import cv2
import matplotlib.pyplot as plt
from shapely.geometry import Polygon

# %matplotlib qt
# %matplotlib tk

def get_visible_face(corners_world, R_wc, t_wc):
    faces = [
        [0,1,2,3],
        [4,5,6,7],
        [0,1,5,4],
        [2,3,7,6],
        [1,2,6,5],
        [0,3,7,4]
    ]

    best_face = None
    best_score = -np.inf

    for face in faces:
        pts = corners_world[face]

        # Compute face normal (in world frame)
        v1 = pts[1] - pts[0]
        v2 = pts[2] - pts[0]
        normal = np.cross(v1, v2)
        normal = normal / np.linalg.norm(normal)
        normal = normal.flatten()

        center = np.mean(pts, axis=0).flatten()

        cam_center = (-R_wc.T @ t_wc).flatten()

        view_dir = center - cam_center
        view_dir = view_dir / np.linalg.norm(view_dir)
        view_dir = view_dir.flatten()

        score = float(np.dot(normal, view_dir))  # <-- FORCE scalar

        if score > best_score:
            best_score = score
            best_face = face

    return best_face

def load_obb(json_path, entity_name):
    with open(json_path, "r") as f:
        data = json.load(f)

    for obj in data:
        if obj["entity"] == entity_name:
            obb = obj["obb"]
            center = np.array(obb["center"])
            extent = np.array(obb["extent"])
            rotation = np.array(obb["rotation"])
            return center, extent, rotation

    raise ValueError(f"{entity_name} not found")

def get_obb_corners(center, extent, R):
    ex, ey, ez = extent / 2

    corners_local = np.array([
        [-ex, -ey, -ez],
        [ ex, -ey, -ez],
        [ ex,  ey, -ez],
        [-ex,  ey, -ez],
        [-ex, -ey,  ez],
        [ ex, -ey,  ez],
        [ ex,  ey,  ez],
        [-ex,  ey,  ez],
    ])

    return center + corners_local @ R.T

def project_points(points, K, R_wc, t_wc):
    projected = []
    valid_mask = []

    for p in points:
        p_cam = (R_wc @ p.reshape(3,1) + t_wc).flatten()
        
        if p_cam[2] <= 0:  # behind camera
            projected.append([np.nan, np.nan])
            valid_mask.append(False)
            continue

        p_img = K @ p_cam
        projected.append(p_img[:2] / p_img[2])
        valid_mask.append(True)

    return np.array(projected), np.array(valid_mask)

def get_front_face(points_world, R_wc, t_wc):
    depths = np.array([(R_wc @ p + t_wc)[2] for p in points_world])
    return np.argsort(depths)[:4]

def get_valid_polygon(corners_2d, valid_mask):
    pts = corners_2d[valid_mask]

    # remove NaNs
    pts = pts[~np.isnan(pts).any(axis=1)]

    if len(pts) < 4:
        return None

    return pts

def compute_iou(poly1, poly2):
    if poly1 is None or poly2 is None:
        return 0.0

    if len(poly1) < 4 or len(poly2) < 4:
        return 0.0

    try:
        p1 = Polygon(poly1).convex_hull
        p2 = Polygon(poly2).convex_hull

        if not p1.is_valid or not p2.is_valid:
            return 0.0

        inter = p1.intersection(p2).area
        union = p1.union(p2).area

        return inter / union if union > 0 else 0

    except:
        return 0.0

def draw_polygon(img, pts, color):
    pts = pts.astype(int)
    for i in range(len(pts)):
        cv2.line(img, tuple(pts[i]), tuple(pts[(i+1)%len(pts)]), color, 2)
        
def compare_obbs(
    pred_center, pred_extent, pred_R,
    gt_center, gt_extent, gt_R,
    image, K, R_wc, t_wc
):
    # --- Corners ---
    pred_corners = get_obb_corners(pred_center, pred_extent, pred_R)
    gt_corners   = get_obb_corners(gt_center, gt_extent, gt_R)

    # --- Projection ---
    pred_2d, pred_mask = project_points(pred_corners, K, R_wc, t_wc)
    gt_2d, gt_mask     = project_points(gt_corners, K, R_wc, t_wc)

    # --- Front face ---
   # --- Select visible face ---
    pred_face_idx = get_visible_face(pred_corners, R_wc, t_wc)
    gt_face_idx   = get_visible_face(gt_corners, R_wc, t_wc)

    # --- Extract polygons ---
    pred_poly = pred_2d[pred_face_idx]
    gt_poly   = gt_2d[gt_face_idx]
    
    # --- IoU ---
    iou = compute_iou(pred_poly, gt_poly)

    # --- Visualization ---
    img_vis = image.copy()
    draw_polygon(img_vis, pred_poly, (0,255,0))  # green = prediction
    draw_polygon(img_vis, gt_poly,   (0,0,255))  # red = ground truth

    return img_vis, iou

gt_center, gt_extent, gt_R = load_obb("Data/sample_answers.json", "vga_socket")


pred_center = socket_pos
pred_extent = extent
pred_R = Rotation_matrix
# pred_center = np.array([
#       0.2696675251269186,
#       0.2269750267617146,
#       0.835096954091098
#     ])
# pred_extent = np.array([
#       0.03778880015958383,
#       0.011160356496244896,
#       0.0060999999999999995
#     ])    
# pred_R =  np.array([[
#         -0.0022095783821239464,
#         0.9990103369723613,
#         0.044423691716723786
#       ],
#       [
#         -0.004352922045486091,
#         0.044413770636362176,
#         -0.9990037382550311
#       ],
#       [
#         -0.9999880848455343,
#         -0.002400749930724931,
#         0.0042504784120087616
#       ]
# 		  ])
# pred_center = np.array([
# 			0.2698147544963437,
# 			0.2270881267849935,
# 			0.8347704194601293
# 		  ])
# pred_extent = np.array([
# 			0.0373324510887077,
# 			0.011448575974808242,
# 			0.0060999999999999995
# 		  ])    
# pred_R =  np.array([[
# 			  0.0001874789799398946,
# 			  0.9989141025023898,
# 			  0.046589491019726835
# 			],
# 			[
# 			  -0.0040196592970269075,
# 			  0.04658986823167808,
# 			  -0.9989060148568988
# 			],
# 			[
# 			  -0.9999919035626079,
# 			  -0.0,
# 			  0.004024028980026629
# 			]
# 		  ])

print("Predicted center:", pred_center)
print("Predicted extent:", pred_extent)
print("Predicted rotation:", pred_R)
R_wc, t_wc = camera_poses[426]

img = cv2.imread("Data/frame_000426.png")

img_vis, iou = compare_obbs(
    pred_center, pred_extent, pred_R,
    gt_center, gt_extent, gt_R,
    img, K, R_wc, t_wc
)

print("IoU:", iou)

# import matplotlib.pyplot as plt
# plt.imshow(cv2.cvtColor(img_vis, cv2.COLOR_BGR2RGB))
# plt.title(f"IoU: {iou:.3f}")
# plt.show()

import matplotlib.pyplot as plt
import cv2

cv2.imshow("OBB Projection", img_vis)
cv2.waitKey(0)
cv2.destroyAllWindows()

Predicted center: [0.28955171 0.22718405 0.52463639]
Predicted extent: [0.00712795 0.01392807 0.00535117]
Predicted rotation: [[-0.03877377 -0.99826172 -0.04438621]
 [ 0.00967822 -0.0447927   0.99894942]
 [ 0.99920114 -0.03830346 -0.01139818]]
IoU: 0.0
